# Notebook 04 - Modeling

**Projet** : Classification des Ast-ro-des Potentiellement Dangereux (PHAs)

Objectif : entra-ner et comparer les mod-les demand-s avec 3 strat-gies de r--quilibrage et une validation crois-e stratifi-e - 5 splits.

Sections :
- A : R-gression Logistique et Arbre de D-cision
- B : Boosting sklearn, substitut robuste - XGBoost/LightGBM si non install-s
- C : SVM lin-aire, avec MLP optionnel disponible dans le module
- Finale : tableau comparatif des 12 configurations

## 0. Imports et configuration

In [1]:
from pathlib import Path
import sys
import importlib.util
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
MODULE_PATH = PROJECT_ROOT / "src" / "ml_phase3.py"
spec = importlib.util.spec_from_file_location("ml_phase3", MODULE_PATH)
ml_phase3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ml_phase3)

TARGET = ml_phase3.TARGET
load_dataset = ml_phase3.load_dataset
make_splits = ml_phase3.make_splits
modeling_configurations = ml_phase3.modeling_configurations
cross_validate_configurations = ml_phase3.cross_validate_configurations
evaluate_on_validation = ml_phase3.evaluate_on_validation

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
print(f"Projet : {PROJECT_ROOT}")

ModuleNotFoundError: No module named 'pandas'

## 1. Chargement du dataset et split stratifi-

Le test set est r-serv- au notebook `06_evaluation.ipynb`. Ici, on utilise le train pour la CV et la validation pour contr-ler le meilleur couple mod-le-strat-gie avant tuning.

In [ ]:
df = load_dataset()
splits = make_splits(df)
X_train, y_train = splits["X_train"], splits["y_train"]
X_val, y_val = splits["X_val"], splits["y_val"]

print(f"Dataset : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print("Distribution cible globale (%):")
display((df[TARGET].value_counts(normalize=True).sort_index() * 100).rename("%"))
print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test r?serv?: {splits['X_test'].shape}")

## 2. Configurations test-es

Les 12 configurations correspondent - 4 familles de mod-les x 3 strat-gies : baseline, pond-ration de classe, oversampling al-atoire.

In [ ]:
configs = modeling_configurations()
configs_df = pd.DataFrame(configs)
display(configs_df)

## 3. Sections A, B, C - Validation crois-e 5 splits

M-trique principale de comparaison : F1 moyen - -cart-type, compl-t- par PR-AUC, ROC-AUC, precision et recall.

Note : `M3_XGBoost` utilise explicitement XGBoost comme demande dans la consigne. Si l import echoue, installez `xgboost` avec conda ou pip.

In [ ]:
modeling_results = cross_validate_configurations(X_train, y_train, configs=configs, n_splits=5)
modeling_results.to_csv(MODELS_DIR / "modeling_results.csv", index=False)
display(modeling_results)

## 4. Tableau comparatif final des 12 configurations

Le meilleur couple mod-le-strat-gie est s-lectionn- selon F1, puis PR-AUC et recall. Cela respecte la logique m-tier : rappel -lev-, mais pas au prix d'une explosion des faux positifs.

In [ ]:
cols = [
    "model", "strategy",
    "f1_mean", "f1_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
    "pr_auc_mean", "pr_auc_std", "roc_auc_mean"
]
comparatif = modeling_results[cols].copy()
display(comparatif.style.background_gradient(subset=["f1_mean", "pr_auc_mean", "recall_mean"], cmap="Greens"))

best_modeling = modeling_results.iloc[0]
print("Meilleur couple CV :")
print(best_modeling[["model", "strategy", "f1_mean", "pr_auc_mean", "recall_mean", "precision_mean"]])

## 5. Contr-le sur validation hold-out

In [ ]:
validation_results = evaluate_on_validation(X_train, y_train, X_val, y_val, modeling_results, top_n=4)
validation_results.to_csv(MODELS_DIR / "validation_results.csv", index=False)
display(validation_results)

## 6. Synth-se modeling

- La baseline logistique fournit une r-f-rence simple et interpr-table.
- L'arbre de d-cision v-rifie une r-gle non lin-aire proche des crit-res NASA.
- Le boosting capture les interactions sans d-pendance externe lourde.
- Le SVM lin-aire couvre la famille MLP/SVM demand-e avec un co-t calcul raisonnable.
- Les r-sultats de ce notebook alimentent `05_tuning.ipynb`.